# recursive-flow graph basics

A tour of the recursive `Graph` API. Every `graph[id]` returns a `Graph` — agents and states are the same primitive at different granularities. The namespaces `graph.all_nodes`, `graph.agents`, and `graph.edges` give explicit access to each. Pure Python — no rendering, no LLM calls.

The fixture under `examples/_data/needle-haystack/` is a saved RecursiveFlow workspace — same on-disk shape any live run produces. For visualizations on top of the same workspace, see [`viz_walkthrough.ipynb`](./viz_walkthrough.ipynb).

**What we cover**

1. Loading a workspace — `Workspace.open_path`, `load_graph`, `load_steps`
2. Navigating agents — `graph.agents[aid]` returns a sub-`Graph`; `graph.children`, `graph.parent_id`
3. Walking nodes — `graph.all_nodes`, `graph.nodes` (root agent's trajectory)
4. Lookups — `graph.all_nodes.find(id)`, `graph[id]` for a sub-`Graph`
5. Filtering — `graph.all_nodes.where`, `graph.all_nodes.errors`, `graph.all_nodes.results`
6. Per-agent results & cost — `graph.result()`, `graph.tokens()`
7. Reading the persisted session directly — `FileSession` + `FileStore`
8. Streaming nodes to sinks — `tee`, `json_logs`

## 1. Load a workspace

A persisted RecursiveFlow run lives in a workspace directory (`graph.json` + `session/` + `context/`). `Workspace.open_path(...)` opens it; `load_graph()` returns the final snapshot, `load_steps()` retraces the whole run as one `Graph` per state-append. We pick the final snapshot for queries against the complete topology; earlier snapshots only contain what had been produced by that point.

In [ ]:
import rflow

ws = rflow.Workspace.open_path("../data/needle-haystack")
graphs = ws.load_steps()
final = graphs[-1]
print(
    f"loaded {len(graphs)} snapshots  \u00b7  final has "
    f"{len(final.agents)} agents, {len(final.all_nodes)} nodes"
)

## 2. Navigating agents

`graph.agents` is a `Mapping[str, Graph]`. `graph.agents["root.boid_js"]` returns the sub-`Graph` rooted at that agent's first node — `nodes` in seq order, `meta` (config, query, system prompt), plus `.children` (a `dict[str, Graph]` of direct child sub-graphs) and `.parent_id` for tree navigation. `graph["..."]` is sugar that resolves either a node id or an agent id.

In [ ]:
print("agents in spawn order:")
for aid, agent in final.agents.items():
    cur = agent.current()
    kind = cur.type if cur else "empty"
    print(f"  {aid:<22} depth={agent.depth}  {kind}")

In [ ]:
print("root.children       :", list(final.children))
print("root.boid_js.parent :", final["root.boid_js"].parent_id)

## 3. Walking nodes

`graph.all_nodes` iterates every node in the subtree in agent / seq order. Useful for counting, scanning, or building summaries. Each per-agent slice is also directly available as `graph.agents[aid].nodes` (or just `graph.nodes` for the current root agent's trajectory).

In [ ]:
print(f"{len(final.all_nodes)} nodes total\n")
for state in final.all_nodes:
    print(f"  {state.agent_id:<22} seq={state.seq:<2} [{state.type}]")

In [ ]:
print("root.boid_js nodes:")
for state in final["root.boid_js"].nodes:
    print(f"  seq={state.seq} [{state.type}]")

## 4. Lookups

Two complementary lookups:

- `graph[ident]` accepts either an *agent id* or a *node id* and returns a sub-`Graph` rooted at that vertex — exactly the same shape as the outer graph.
- `graph.all_nodes.find(node_id)` returns the bare `Node` (no sub-rooting) — handy when you just want the payload.

In [ ]:
hit_agent = final["root.boid_js"]
print(f"agent: {hit_agent.agent_id} \u2192 result: {(hit_agent.result() or '')[:80]!r}")

some_node = list(final.all_nodes)[0]
hit_node = final.all_nodes.find(some_node.id)
print(f"state: {hit_node.agent_id} [{hit_node.type}] seq={hit_node.seq}")

## 5. Filtering

`graph.all_nodes` carries the same selectors as the engine's internals. Convenience: `graph.all_nodes.results()` (`ResultNode`s), `graph.all_nodes.errors()` (`ErrorNode`s), `graph.all_nodes.supervising()`, etc. `graph.all_nodes.where(...)` is the general case — keyword args match attributes, or pass a callable predicate.

In [ ]:
print("results :", [(e.agent_id, e.result[:40]) for e in final.all_nodes.results()])
print("errors  :", final.all_nodes.errors() or "(none)")

In [ ]:
# kwargs match attributes
child_llm = final.all_nodes.where(type="llm_action", agent_id="root.boid_js")
print("via kwargs    :", [(e.agent_id, e.type, e.seq) for e in child_llm])

# or pass a predicate
delegating = final.all_nodes.where(
    lambda e: e.type == "llm_output" and "delegate(" in getattr(e, "code", "")
)
print("via predicate :", [(e.agent_id, e.seq) for e in delegating])

## 6. Per-agent results & cost

Each sub-`Graph` aggregates its own nodes: `graph.result()` returns the latest `ResultNode` payload, `graph.tokens()` sums input/output tokens across the subtree (`recursive=False` for just this agent). At the top level, `final.tokens()` totals the whole run.

In [ ]:
for aid, agent in final.agents.items():
    inp, out = agent.tokens(recursive=False)
    res = (agent.result() or "")[:60]
    print(f"  {aid:<22} {inp + out:>5} tok  \u2192 {res!r}")

total_in, total_out = final.tokens()
print(f"\ntotal: {total_in + total_out:,} ({total_in:,} in / {total_out:,} out)")

## 7. Reading the persisted session directly

`Workspace.session` persists every state to disk as it's written, and `Workspace.open_path(...).load_graph()` is the high-level entrypoint we used in section 1. If you want to skip even constructing a `Workspace` (e.g. you only need the graph), you can load straight off the store with `FileSession` + `FileStore` — same `Graph` shape, no extra wiring.

In [ ]:
from pathlib import Path

import rflow

workspace_root = Path("../data/needle-haystack")
session = rflow.FileSession(rflow.FileStore(workspace_root))
persisted = session.load_graph()
print(
    f"persisted {len(persisted.agents)} agents, "
    f"{len(persisted.all_nodes)} nodes\n"
)
print(persisted.tree())

## 8. Streaming nodes to sinks

`tee(stream, *sinks)` fans a graph iterator out to multiple callables — useful during a live run for writing to a file, posting to Slack, pushing to a UI, etc. `json_logs(graph, path)` writes a newline-delimited JSON state log for external tools (Loki, Datadog, DuckDB).

In [ ]:
from rflow.utils.viz import tee

captured = []
for _ in tee(graphs, captured.append):
    pass
print(f"tee forwarded {len(captured)} snapshots to a sink")

In [ ]:
import tempfile
from rflow.utils.tracing import json_logs

with tempfile.NamedTemporaryFile(suffix=".jsonl", delete=False) as f:
    out = json_logs(final, f.name)
lines = out.read_text().splitlines()
print(f"json_logs wrote {len(lines)} lines to {out}")
print("first line preview:")
print(lines[0][:200] + "...")